# In vitro decidualization bulk RNA-seq: DEGs at 3, 6 and 8 days of cAMP + MPA

| Timepoint | Study | Treated | Control |
|---|---|---|---|
| 3d | Rytkonen et al. 2019 (GSE114296) | DSC3d (n = 3) | ESF (n = 3) |
| 6d | Stadtmauer & Wagner 2022 (PRJNA679324) | cAMP + MPA (n = 2) | base media (n = 2) |
| 8d | Rytkonen et al. 2019 (GSE114296) | DSC8d (n = 2) | ESF (n = 3) |

Single-end FASTQs -> FastQC/MultiQC -> kallisto (Ensembl 110, matching Cell Ranger GRCh38-2024-A)
-> gene-level counts/TPM -> PyDESeq2 -> in vitro core DEGs (same direction at 3d, 6d and 8d).

Every step skips work that is already done (existing FastQC reports, kallisto index, quantifications),
so re-running the notebook only redoes the analysis.

**Note on Rytkonen 2019:** three SRA runs are linked to the wrong GEO sample. Based on decidual markers
and the authors' own count table, ESF_1, ESF_2 and DSC3days_3 are decidualized (3d) and ESF_3,
DSC3days_1 and DSC3days_2 are controls. The groups below use the corrected assignment; folder names
keep the GEO labels.

**Outputs** (in `OUT_DIR`):
* `<study>_gene_counts.csv`, `<study>_gene_tpm.csv` - gene-level counts and TPM per study
* `<timepoint>_DEGs.csv` and `<timepoint>_all_genes.csv` for each contrast
* `InVitroCoreDEGs.csv` - DEGs shared by 3d, 6d and 8d, with log2FC / p / padj at each timepoint
* `all_genes_merged.tsv` - every gene at all three timepoints (input for the single-cell notebook)
* `InVitro_sample_PCA.svg`, `InVitro_DEG_intersections.svg`, `InVitro_lfc_agreement.svg`

FastQC and MultiQC reports are written to `<study>/qc/`, kallisto output to `<study>/quant/`.

## 1. Libraries

In [ ]:
import gzip
import re
import subprocess
import sys
import urllib.request
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

plt.rcParams["svg.fonttype"] = "none"
pd.set_option("display.width", 200)

BIN = Path(sys.executable).parent

## 2. Datasets

`samples`: sample name (= kallisto output folder) -> (group, FASTQ files).
A sample with several FASTQs is quantified as one (runs merged).

In [ ]:
BASE = Path("/directoryto/BulkRnaSeq")

DATASETS = {
    "Stadtmauer2022": dict(
        dir=BASE / "Stadtmauer2022",
        groups=["base", "cAMP_MPA"],                   # first = reference level
        samples={
            "SRR13081989": ("base",     ["SRR13081989.fastq.gz"]),
            "SRR13081990": ("base",     ["SRR13081990.fastq.gz"]),
            "SRR13081985": ("cAMP_MPA", ["SRR13081985.fastq.gz"]),
            "SRR13081986": ("cAMP_MPA", ["SRR13081986.fastq.gz"]),
        },
    ),
    "Rytkonen2019": dict(
        dir=BASE / "Rytkonen2019",
        groups=["ESF", "DSC3d", "DSC8d"],
        samples={                                      # corrected groups (see note above)
            "ESF_1":      ("DSC3d", ["SRR7142614.fastq.gz"]),
            "ESF_2":      ("DSC3d", ["SRR7142615.fastq.gz"]),
            "ESF_3":      ("ESF",   ["SRR7142616.fastq.gz"]),
            "DSC3days_1": ("ESF",   ["SRR7142617.fastq.gz"]),
            "DSC3days_2": ("ESF",   ["SRR7142618.fastq.gz"]),
            "DSC3days_3": ("DSC3d", ["SRR7142619.fastq.gz"]),
            "DSC8days_1": ("DSC8d", ["SRR7142620.fastq.gz"]),   # file holds both runs (620 + 621)
            "DSC8days_2": ("DSC8d", ["SRR7142622.fastq.gz"]),   # file holds both runs (622 + 623)
        },
    ),
}

# timepoint -> (dataset, treated group, control group)
CONTRASTS = {
    "3d": ("Rytkonen2019",   "DSC3d",    "ESF"),
    "6d": ("Stadtmauer2022", "cAMP_MPA", "base"),
    "8d": ("Rytkonen2019",   "DSC8d",    "ESF"),
}

for name, ds in DATASETS.items():
    missing = [f for _, (_, fq) in ds["samples"].items() for f in fq if not (ds["dir"] / f).exists()]
    print(f"{name}: {len(ds['samples'])} samples, groups {ds['groups']}"
          + (f"  MISSING: {missing}" if missing else "  (all FASTQs present)"))

## 3. Variables

In [ ]:
THREADS = 8

# Reference: Ensembl 110 = Cell Ranger GRCh38-2024-A (GENCODE v44)
ENSEMBL_RELEASE = 110
REF_DIR = BASE / "human_reference" / f"ensembl{ENSEMBL_RELEASE}"
TX_FASTA = REF_DIR / "GRCh38_cdna_ncrna.fa.gz"
INDEX = REF_DIR / "GRCh38_cdna_ncrna.idx"

FRAG_LEN, FRAG_SD = 200, 20          # single-end reads: kallisto fragment length estimate

MIN_COUNT, MIN_SAMPLES = 10, 2       # prefilter: >= MIN_COUNT counts in >= MIN_SAMPLES samples
FDR, FC, MIN_TPM = 0.01, 2.0, 3.0    # DEG cutoffs: padj <= 0.01, |FC| >= 2, mean TPM > 3 (same as single-cell)

OUT_DIR = BASE / "SharedDEGs_strict"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MARKERS = ["PRL", "IGFBP1", "FOXO1", "HAND2", "WNT4", "LEFTY2", "PTGER2", "PTGER4", "POSTN"]


def run(cmd, log=None):
    """Run a command from this kernel's environment; stop with its output if it fails."""
    cmd = [str(BIN / cmd[0]) if (BIN / cmd[0]).exists() else cmd[0]] + [str(c) for c in cmd[1:]]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if log:
        Path(log).parent.mkdir(parents=True, exist_ok=True)
        Path(log).write_text(p.stdout + p.stderr)
    if p.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{p.stderr[-2000:]}")
    return p


print(REF_DIR)
print("FASTA found:", TX_FASTA.exists(), "| index found:", INDEX.exists())

## 4. Quality control (FastQC + MultiQC)

In [ ]:
for name, ds in DATASETS.items():
    qc = ds["dir"] / "qc" / "fastqc"
    qc.mkdir(parents=True, exist_ok=True)
    fastqs = [ds["dir"] / f for _, (_, fq) in ds["samples"].items() for f in fq]
    todo = [f for f in fastqs if not (qc / f.name.replace(".fastq.gz", "_fastqc.html")).exists()]
    if todo:
        print(f"{name}: FastQC on {len(todo)} files ...")
        run(["fastqc", "-t", THREADS, "--outdir", qc] + todo)
    else:
        print(f"{name}: FastQC already done")

## 5. Reference and kallisto quantification

In [ ]:
# Reference transcriptome + index (built once)
REF_DIR.mkdir(parents=True, exist_ok=True)
if not TX_FASTA.exists():
    url = f"https://ftp.ensembl.org/pub/release-{ENSEMBL_RELEASE}/fasta/homo_sapiens"
    with open(TX_FASTA, "wb") as out:
        for part in ["cdna/Homo_sapiens.GRCh38.cdna.all.fa.gz", "ncrna/Homo_sapiens.GRCh38.ncrna.fa.gz"]:
            print("Downloading", part)
            with urllib.request.urlopen(f"{url}/{part}") as r:
                out.write(r.read())
if not INDEX.exists():
    print("Building kallisto index (10-30 min) ...")
    run(["kallisto", "index", "-t", THREADS, "-i", INDEX, TX_FASTA])
print(f"Index: {INDEX}")

# Quantify
for name, ds in DATASETS.items():
    print(f"\n{name}")
    for sample, (group, fq) in ds["samples"].items():
        outdir = ds["dir"] / "quant" / sample
        if (outdir / "abundance.tsv").exists():
            print(f"  {sample:12s} already quantified")
            continue
        print(f"  {sample:12s} kallisto quant ...")
        outdir.mkdir(parents=True, exist_ok=True)
        run(["kallisto", "quant", "-i", INDEX, "-o", outdir, "--single", "-l", FRAG_LEN, "-s", FRAG_SD,
             "-t", THREADS] + [ds["dir"] / f for f in fq],
            log=ds["dir"] / "quant" / f"{sample}.kallisto.log")

# MultiQC (FastQC + kallisto logs)
for name, ds in DATASETS.items():
    run(["multiqc", "-f", "-o", ds["dir"] / "qc", ds["dir"] / "qc" / "fastqc", ds["dir"] / "quant"])
    print(f"{name}: MultiQC report -> {ds['dir'] / 'qc' / 'multiqc_report.html'}")

## 6. Gene-level counts and TPM

In [ ]:
# transcript -> gene map from the Ensembl FASTA headers
rows = []
with gzip.open(TX_FASTA, "rt") as fh:
    for line in fh:
        if line.startswith(">"):
            tx = line[1:].split()[0].split(".")[0]
            g = re.search(r"gene:(\S+)", line)
            s = re.search(r"gene_symbol:(\S+)", line)
            b = re.search(r"gene_biotype:(\S+)", line)
            rows.append((tx, g.group(1).split(".")[0] if g else tx, s.group(1) if s else "",
                         b.group(1) if b else ""))
tx2gene = pd.DataFrame(rows, columns=["transcript_id", "gene_id", "gene_symbol", "gene_biotype"]) \
            .drop_duplicates("transcript_id").set_index("transcript_id")
GENE_INFO = tx2gene.drop_duplicates("gene_id").set_index("gene_id")[["gene_symbol", "gene_biotype"]]
print(f"{len(tx2gene):,} transcripts, {len(GENE_INFO):,} genes")

COUNTS, TPM, META = {}, {}, {}
for name, ds in DATASETS.items():
    cnt, tpm, mapped = {}, {}, {}
    for sample in ds["samples"]:
        ab = pd.read_csv(ds["dir"] / "quant" / sample / "abundance.tsv", sep="\t")
        ab["gene_id"] = ab["target_id"].str.split(".").str[0].map(tx2gene["gene_id"])
        ab = ab.dropna(subset=["gene_id"])
        cnt[sample] = ab.groupby("gene_id")["est_counts"].sum()
        tpm[sample] = ab.groupby("gene_id")["tpm"].sum()
        log = ds["dir"] / "quant" / f"{sample}.kallisto.log"
        m = re.search(r"processed ([\d,]+) reads, ([\d,]+) reads pseudoaligned", log.read_text()) if log.exists() else None
        mapped[sample] = (int(m.group(2).replace(",", "")) / int(m.group(1).replace(",", "")) * 100) if m else np.nan
    COUNTS[name] = pd.DataFrame(cnt).fillna(0)
    TPM[name] = pd.DataFrame(tpm).fillna(0)
    META[name] = pd.DataFrame({"group": [ds["samples"][s][0] for s in ds["samples"]],
                               "pct_pseudoaligned": [mapped[s] for s in ds["samples"]]},
                              index=list(ds["samples"]))
    COUNTS[name].join(GENE_INFO).to_csv(OUT_DIR / f"{name}_gene_counts.csv")
    TPM[name].join(GENE_INFO).to_csv(OUT_DIR / f"{name}_gene_tpm.csv")
    print(f"\n{name}: {COUNTS[name].shape[0]:,} genes x {COUNTS[name].shape[1]} samples")
    print(META[name].assign(million_counts=(COUNTS[name].sum() / 1e6).round(1)).round(1).to_string())

## 7. Sample QC: decidual markers, correlation and PCA

Every sample should match its group: markers such as IGFBP1, FOXO1 and PRL up in treated samples,
POSTN and PTGER4 down. This is the check that exposed the Rytkonen labelling error.

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(5.5 * len(DATASETS), 4.5))
for ax, (name, ds) in zip(np.atleast_1d(axes), DATASETS.items()):
    meta, tpm, cnt = META[name], TPM[name], COUNTS[name]
    order = meta.sort_values("group", key=lambda g: g.map({k: i for i, k in enumerate(ds["groups"])})).index

    sym = GENE_INFO["gene_symbol"]
    mk = tpm.loc[tpm.index.intersection(sym[sym.isin(MARKERS)].index), order]
    mk.index = sym[mk.index]
    mk.columns = [f"{s}\n({meta.loc[s, 'group']})" for s in order]
    print(f"\n{name} - marker TPM")
    print(mk.loc[[m for m in MARKERS if m in mk.index]].round(1).to_string())

    cpm = cnt / cnt.sum() * 1e6
    logcpm = np.log2(cpm[(cpm >= 1).sum(axis=1) >= 2] + 1)
    print(f"\n{name} - sample correlation (log2 CPM)")
    print(logcpm[order].corr().round(3).to_string())

    top = logcpm.var(axis=1).sort_values(ascending=False).index[:500]
    x = logcpm.loc[top, order].T.values
    x = x - x.mean(axis=0)
    u, s, _ = np.linalg.svd(x, full_matrices=False)
    pcs, var = u * s, s ** 2 / (s ** 2).sum() * 100
    colors = dict(zip(ds["groups"], ["#7f8c8d", "#e67e22", "#c0392b"]))
    for i, smp in enumerate(order):
        g = meta.loc[smp, "group"]
        ax.scatter(pcs[i, 0], pcs[i, 1], c=colors[g], s=50,
                   label=g if g not in ax.get_legend_handles_labels()[1] else None)
        ax.annotate(smp, (pcs[i, 0], pcs[i, 1]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel(f"PC1 ({var[0]:.1f}%)"); ax.set_ylabel(f"PC2 ({var[1]:.1f}%)")
    ax.set_title(name, fontsize=10); ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "InVitro_sample_PCA.svg", bbox_inches="tight")
plt.show()

## 8. PyDESeq2

One model per study (`~group`); each timepoint is a contrast against its control.
DEG: padj <= `FDR`, |FC| >= `FC`, mean TPM > `MIN_TPM` in the treated or control group.

In [ ]:
DDS, RES = {}, {}
for name, ds in DATASETS.items():
    cnt = COUNTS[name].round().astype(int)
    cnt = cnt[(cnt >= MIN_COUNT).sum(axis=1) >= MIN_SAMPLES]
    meta = META[name][["group"]].copy()
    meta["group"] = pd.Categorical(meta["group"], categories=ds["groups"])
    inference = DefaultInference(n_cpus=THREADS)
    dds = DeseqDataSet(counts=cnt.T, metadata=meta, design="~group", refit_cooks=True,
                       inference=inference, quiet=True)
    dds.deseq2()
    DDS[name] = (dds, inference)
    print(f"{name}: {cnt.shape[0]:,} genes tested, {len(meta)} samples")

lfc_cut = np.log2(FC)
for tp, (name, test, ref) in CONTRASTS.items():
    dds, inference = DDS[name]
    ds = DeseqStats(dds, contrast=["group", test, ref], inference=inference, quiet=True)
    ds.summary()
    res = ds.results_df.copy()
    if ref == DATASETS[name]["groups"][0]:                    # shrinkage needs a model coefficient
        ds.lfc_shrink(coeff=f"group[T.{test}]")
        res["log2FoldChange_shrunk"] = ds.results_df["log2FoldChange"]

    groups = META[name]["group"]
    for lvl in (ref, test):
        res[f"meanTPM_{lvl}"] = TPM[name].loc[res.index, groups.index[groups == lvl]].mean(axis=1)
    expressed = res[[f"meanTPM_{ref}", f"meanTPM_{test}"]].max(axis=1) > MIN_TPM

    sig = (res["padj"] <= FDR) & expressed
    res["direction"] = "ns"
    res.loc[sig & (res["log2FoldChange"] >= lfc_cut), "direction"] = "up"
    res.loc[sig & (res["log2FoldChange"] <= -lfc_cut), "direction"] = "down"
    res = GENE_INFO.reindex(res.index).join(res).sort_values("padj")
    res.index.name = "gene_id"
    RES[tp] = res
    res.to_csv(OUT_DIR / f"{tp}_all_genes.csv")
    res[res.direction != "ns"].to_csv(OUT_DIR / f"{tp}_DEGs.csv")

    print(f"\n{tp}  ({name}: {test} vs {ref})  up {(res.direction == 'up').sum()}, "
          f"down {(res.direction == 'down').sum()}")
    print(res[res.gene_symbol.isin(MARKERS)][["gene_symbol", "log2FoldChange", "padj", "direction"]].to_string())
print(f"\nCutoffs: padj <= {FDR}, |FC| >= {FC}, mean TPM > {MIN_TPM}")

## 9. Shared in vitro DEGs (3d, 6d, 8d)

Only genes tested at all three timepoints are compared. The in vitro core = DEG in the same direction at 3d, 6d and 8d.

In [ ]:
TP = list(CONTRASTS)
parts = []
for tp in TP:
    r = RES[tp]
    p = pd.DataFrame({f"log2FC_{tp}": r["log2FoldChange"], f"pvalue_{tp}": r["pvalue"],
                      f"padj_{tp}": r["padj"],
                      f"maxTPM_{tp}": r.filter(like="meanTPM_").max(axis=1),
                      f"dir_{tp}": r["direction"]})
    if "log2FoldChange_shrunk" in r:
        p[f"log2FC_shrunk_{tp}"] = r["log2FoldChange_shrunk"]
    parts.append(p)
merged = GENE_INFO.join(pd.concat(parts, axis=1, join="inner"), how="inner")

dirs = merged[[f"dir_{tp}" for tp in TP]]
merged["pattern"] = dirs.apply(lambda r: "/".join(f"{t}:{v}" for t, v in zip(TP, r)), axis=1)
merged["mean_log2FC"] = merged[[f"log2FC_{tp}" for tp in TP]].mean(axis=1)
merged["invitro_core"] = np.where((dirs == "up").all(axis=1), "up",
                          np.where((dirs == "down").all(axis=1), "down", "no"))
merged.index.name = "gene_id"

core = merged[merged.invitro_core != "no"].sort_values(["invitro_core", "mean_log2FC"], ascending=[False, False])
core_cols = ["gene_symbol", "gene_biotype", "invitro_core"] + \
            [f"{k}_{tp}" for tp in TP for k in ("log2FC", "pvalue", "padj")]
core[core_cols].to_csv(OUT_DIR / "InVitroCoreDEGs.csv")
merged.to_csv(OUT_DIR / "all_genes_merged.tsv", sep="\t")        # read by the single-cell notebook

print(f"Genes tested at all three timepoints: {len(merged):,}")
for tp in TP:
    print(f"  {tp}: up {(merged[f'dir_{tp}'] == 'up').sum()}, down {(merged[f'dir_{tp}'] == 'down').sum()}")
print(f"In vitro core: up {(core.invitro_core == 'up').sum()}, down {(core.invitro_core == 'down').sum()}")
print(f"Discordant (up at one timepoint, down at another): "
      f"{(dirs.isin(['up']).any(axis=1) & dirs.isin(['down']).any(axis=1)).sum()}")
print(f"\nSaved {OUT_DIR / 'InVitroCoreDEGs.csv'} and {OUT_DIR / 'all_genes_merged.tsv'}")

## 10. Plot Results

In [ ]:
# DEG intersections (exclusive) for up- and downregulated genes
combos = [c for r in range(len(TP), 0, -1) for c in combinations(TP, r)]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, d, color in zip(axes, ["up", "down"], ["#c0392b", "#2471a3"]):
    flags = pd.DataFrame({t: merged[f"dir_{t}"] == d for t in TP})
    sizes = [int((flags[list(c)].all(axis=1) & ~flags[[t for t in TP if t not in c]].any(axis=1)).sum())
             for c in combos]
    ax.bar(range(len(combos)), sizes, color=color, width=0.65)
    for i, s in enumerate(sizes):
        ax.text(i, s, str(s), ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(len(combos)))
    ax.set_xticklabels([" & ".join(c) + (" only" if len(c) == 1 else "") for c in combos],
                       rotation=35, ha="right", fontsize=8)
    ax.set_title(f"{d}regulated: " + ", ".join(f"{t} {int(flags[t].sum())}" for t in TP), fontsize=9)
    ax.set_ylabel("genes")
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "InVitro_DEG_intersections.svg", bbox_inches="tight")
plt.show()

# Pairwise log2FC agreement between timepoints
pairs = list(combinations(TP, 2))
fig, axes = plt.subplots(1, len(pairs), figsize=(4.2 * len(pairs), 4))
for ax, (a, b) in zip(axes, pairs):
    ca = f"log2FC_shrunk_{a}" if f"log2FC_shrunk_{a}" in merged else f"log2FC_{a}"
    cb = f"log2FC_shrunk_{b}" if f"log2FC_shrunk_{b}" in merged else f"log2FC_{b}"
    ok = merged[[ca, cb, "invitro_core"]].dropna()
    hit = ok.invitro_core != "no"
    ax.scatter(ok.loc[~hit, ca], ok.loc[~hit, cb], s=2, c="#bbbbbb", linewidths=0)
    ax.scatter(ok.loc[hit, ca], ok.loc[hit, cb], s=3, c="#8e44ad", linewidths=0, label="in vitro core")
    lim = np.nanpercentile(np.abs(ok[[ca, cb]].values), 99.9)
    ax.plot([-lim, lim], [-lim, lim], lw=0.6, c="k", ls="--")
    ax.axhline(0, lw=0.4, c="k"); ax.axvline(0, lw=0.4, c="k")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel(f"{a} log2FC"); ax.set_ylabel(f"{b} log2FC")
    ax.set_title(f"{a} vs {b}   r = {ok[[ca, cb]].corr().iloc[0, 1]:.2f}", fontsize=9)
axes[0].legend(frameon=False, fontsize=7, markerscale=4)
fig.tight_layout()
fig.savefig(OUT_DIR / "InVitro_lfc_agreement.svg", bbox_inches="tight")
plt.show()
print(f"Figures saved in {OUT_DIR}/")